In [3]:
import json
import pandas as pd
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
GENOME_DIR = REPO_ROOT / "data/raw/book-genome/book_dataset"
PROCESSED = REPO_ROOT / "data/processed"


In [4]:
# --- 1. Load genome data ---
print("Loading genome scores (tagdl)...")
scores = pd.read_csv(GENOME_DIR / "scores" / "tagdl.csv")
print(f"  {len(scores):,} (tag, item_id, score) rows")
print(f"  {scores['item_id'].nunique():,} unique books in genome")
print(f"  {scores['tag'].nunique():,} unique tags")
print(f"  Score range: {scores['score'].min():.3f} to {scores['score'].max():.3f}")
print()

Loading genome scores (tagdl)...
  6,814,898 (tag, item_id, score) rows
  9,374 unique books in genome
  727 unique tags
  Score range: -0.208 to 1.188



In [5]:
# --- 2. Load metadata (need to map their item_id to titles, then to OUR book_id) ---
print("Loading genome metadata...")
metadata_rows = []
with open(GENOME_DIR / "raw" / "metadata.json", "r", encoding="utf-8") as f:
    for line in f:
        metadata_rows.append(json.loads(line))
meta = pd.DataFrame(metadata_rows)
print(f"  {len(meta):,} books in metadata")
print(f"  Columns: {meta.columns.tolist()}")
print(f"  Sample:")
print(meta[["item_id", "title", "authors", "year"]].head())
print()

Loading genome metadata...
  9,374 books in metadata
  Columns: ['item_id', 'url', 'title', 'authors', 'lang', 'img', 'year', 'description']
  Sample:
    item_id                                    title          authors  year
0  16827462                   The Fault in Our Stars       John Green  2012
1   2792775  The Hunger Games (The Hunger Games, #1)  Suzanne Collins  2008
2   8812783        Mockingjay (The Hunger Games, #3)  Suzanne Collins  2010
3  41107568                    The Girl on the Train    Paula Hawkins  2015
4   6171458     Catching Fire (The Hunger Games, #2)  Suzanne Collins  2009



In [7]:
# Diagnostic: compare actual values
print("Their item_id:")
print(f"  dtype: {meta['item_id'].dtype}")
print(f"  sample: {meta['item_id'].head(5).tolist()}")

print("\nOur book_id:")
print(f"  dtype: {df['book_id'].dtype}")
print(f"  sample: {df['book_id'].head(5).tolist()}")

# Try direct match using a known book — Fault in Our Stars, their item_id is 16827462
test_id_int = 16827462
test_id_str = "16827462"

match_int = df[df['book_id'] == test_id_int]
match_str = df[df['book_id'] == test_id_str]
match_contains = df[df['title'].str.contains("Fault in Our Stars", case=False, na=False)]

print(f"\nLooking for 'The Fault in Our Stars' (their item_id={test_id_int}):")
print(f"  Match on book_id == int: {len(match_int)} rows")
print(f"  Match on book_id == str: {len(match_str)} rows")
print(f"  Match on title contains: {len(match_contains)} rows")
if len(match_contains):
    print(f"  In our corpus, this book has book_id: {match_contains.iloc[0]['book_id']} (type {type(match_contains.iloc[0]['book_id']).__name__})")

Their item_id:
  dtype: int64
  sample: [16827462, 2792775, 8812783, 41107568, 6171458]

Our book_id:
  dtype: str
  sample: ['6066819', '89376', '89378', '6158967', '18628482']

Looking for 'The Fault in Our Stars' (their item_id=16827462):
  Match on book_id == int: 0 rows
  Match on book_id == str: 0 rows
  Match on title contains: 2 rows
  In our corpus, this book has book_id: 18717927 (type str)


In [8]:
# Match by normalized title + first author
import re

def norm(s):
    if not isinstance(s, str): return ""
    return re.sub(r'[^a-z0-9]', '', s.lower())

# Our corpus key
df["match_key"] = df["title"].apply(norm) + "|" + df["primary_author"].apply(lambda x: norm(x).split("|")[0] if x else "")

# Their corpus key — they have 'authors' as a string, may include multiple
def first_author(a):
    if not isinstance(a, str): return ""
    # Authors field may be comma-separated or pipe-separated; grab first
    parts = re.split(r'[,|/]', a)
    return norm(parts[0]) if parts else ""

meta["match_key"] = meta["title"].apply(norm) + "|" + meta["authors"].apply(first_author)

# Check overlap
our_keys = set(df["match_key"])
their_keys = set(meta["match_key"])
overlap = our_keys & their_keys

print(f"Genome books: {len(meta):,}")
print(f"Matched to our corpus by title+author: {len(overlap):,}")
print(f"  ({len(overlap)/len(meta)*100:.1f}% of genome books found)")

# Coverage of recommendable subset
recommendable = df[df["ratings_count"] >= 2000]
rec_keys = set(recommendable["match_key"])
rec_overlap = rec_keys & their_keys
print(f"\nRecommendable books (>=2000 ratings): {len(recommendable):,}")
print(f"  With genome scores: {len(rec_overlap):,}")
print(f"  Coverage: {len(rec_overlap)/len(recommendable)*100:.1f}%")

# Spot check Fault in Our Stars
fios = norm("The Fault in Our Stars") + "|" + norm("John Green")
print(f"\nFault in Our Stars match: {fios in overlap}")

Genome books: 9,374
Matched to our corpus by title+author: 9,199
  (98.1% of genome books found)

Recommendable books (>=2000 ratings): 58,092
  With genome scores: 8,754
  Coverage: 15.1%

Fault in Our Stars match: True
